In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

c:\Users\sripa\miniconda3\envs\charlie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sripa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sripa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sripa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
660,Where was this film when I was a kid? After hi...,positive
321,I liked SOLINO very much. It is a very heart-r...,positive
524,This movie isn't about football at all. It's a...,negative
509,Mickey Rourke is enjoying a renaissance at the...,negative
803,"As a South African, living in South Africa aga...",negative


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [5]:
df = normalize_text(df)
df.head()

,review,sentiment
660,film kid parent split tadashi move mom live gr...,positive
321,liked solino much heart rending story italian ...,positive
524,movie football all jesus god kind sappy sancti...,negative
509,mickey rourke enjoying renaissance moment fair...,negative
803,south african living south africa year stay uk...,negative


In [6]:
df['sentiment'].value_counts()

sentiment
negative    270
positive    230
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
660,film kid parent split tadashi move mom live gr...,1
321,liked solino much heart rending story italian ...,1
524,movie football all jesus god kind sappy sancti...,0
509,mickey rourke enjoying renaissance moment fair...,0
803,south african living south africa year stay uk...,0


In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [27]:
mf = 50
testsize = 0.3

In [28]:
vectorizer = CountVectorizer(max_features=mf)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=testsize, random_state=42)

In [30]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/PatilSri585/MLOPS-Capstone.mlflow')
dagshub.init(repo_owner='PatilSri585', repo_name='MLOPS-Capstone', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline Initial Experiment")


2026-03-25 17:52:15,246 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/PatilSri585/MLOPS-Capstone "HTTP/1.1 200 OK"


Initialized MLflow to track repo "PatilSri585/MLOPS-Capstone"

2026-03-25 17:52:15,254 - INFO - Initialized MLflow to track repo "PatilSri585/MLOPS-Capstone"


Repository PatilSri585/MLOPS-Capstone initialized!

2026-03-25 17:52:15,259 - INFO - Repository PatilSri585/MLOPS-Capstone initialized!


<Experiment: artifact_location='mlflow-artifacts:/20611c74481f40ba9f2bf3fd959a1b3a', creation_time=1774441109061, experiment_id='1', last_update_time=1774441109061, lifecycle_stage='active', name='Logistic Regression Baseline Initial Experiment', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [31]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", mf)
        mlflow.log_param("test_size", testsize)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-03-25 17:52:15,773 - INFO - Starting MLflow run...
2026-03-25 17:52:16,333 - INFO - Logging preprocessing parameters...
2026-03-25 17:52:17,396 - INFO - Initializing Logistic Regression model...
2026-03-25 17:52:17,399 - INFO - Fitting the model...
2026-03-25 17:52:17,460 - INFO - Model training complete.
2026-03-25 17:52:17,462 - INFO - Logging model parameters...
2026-03-25 17:52:17,781 - INFO - Making predictions...
2026-03-25 17:52:17,783 - INFO - Calculating evaluation metrics...
2026-03-25 17:52:17,806 - INFO - Logging evaluation metrics...
2026-03-25 17:52:19,288 - INFO - Saving and logging the model...
2026/03/25 17:52:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 17:52:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run glamorous-quail-107 at: https://dagshub.com/PatilSri585/MLOPS-Capstone.mlflow/#/experiments/1/runs/c738880943754ffe803a5385d5a43cee
🧪 View experiment at: https://dagshub.com/PatilSri585/MLOPS-Capstone.mlflow/#/experiments/1
